In [10]:
%pip cache purge

%pip install -r ../requirements.txt


Files removed: 0 (0 bytes)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import shutil

DIRECTORIES = [
    "../models", 
    "../data/raw/files"
]
deleted_files = []

for directory in DIRECTORIES:
    if not os.path.exists(directory):
        print(f"Directory '{directory}' does not exist.")
        continue

    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)

        if item == ".gitkeep":
            continue  # Skip .gitkeep

        if os.path.isfile(item_path):
            os.remove(item_path)
            deleted_files.append(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            deleted_files.append(item_path + "/")  

if deleted_files:
    print("Deleted files and directories:")
    for file in deleted_files:
        print(f" - {file}")
else:
    print("No files to delete.")


Deleted files and directories:
 - ../data/raw/files/MNE-eegbci-data/


In [ ]:
import numpy as np
import pandas as pd
import random
import os
import logging
import warnings
from datetime import datetime

# Librerías para EEG
import mne
from mne.datasets import eegbci
from mne.io import read_raw_edf, concatenate_raws

# Procesamiento de señales
from scipy import signal

# Configuración
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('eeg_preprocessing')

# Ignorar warnings
warnings.filterwarnings('ignore')

# Parámetros de preprocesamiento
PREPROCESSING_PARAMS = {
    'low_cutoff': 4,      # Hz - Incluimos ondas theta (4-8 Hz)
    'high_cutoff': 45,    # Hz - Incluimos hasta gama bajo (30-45 Hz)
    'apply_notch': True,  # Filtro notch para ruido de línea eléctrica
    'tmin': -1.0,         # Tiempo inicial para épocas (segundos)
    'tmax': 4.0,          # Tiempo final para épocas (segundos)
    'csp_components': 6   # Componentes CSP a utilizar
}

# Función para cargar un sujeto aleatorio
def load_random_subject(exclude_subjects=None):
    """
    Carga datos EEG de un sujeto aleatorio, excluyendo los sujetos especificados.
    
    Args:
        exclude_subjects (list): Lista de IDs de sujetos a excluir
        
    Returns:
        tuple: (raw_data, subject_id, run_id, task_type, paradigm)
    """
    # Inicializar lista de exclusión si es None
    if exclude_subjects is None:
        exclude_subjects = []
    
    # Determinar sujetos disponibles (1-109, excluyendo los ya seleccionados)
    available_subjects = [s for s in range(1, 110) if s not in exclude_subjects]
    
    if not available_subjects:
        raise ValueError("No hay sujetos disponibles para seleccionar")
    
    # Seleccionar un sujeto aleatorio
    subject = random.choice(available_subjects)
    
    # Definir los tipos de runs disponibles
    motor_execution_runs = [3, 5, 7, 9, 11, 13]  # Izquierda/derecha o manos/pies
    motor_imagery_runs = [4, 6, 8, 10, 12, 14]   # Izquierda/derecha o manos/pies
    
    # Seleccionar aleatoriamente entre ejecución motora o imaginación motora
    if random.choice([True, False]):
        runs_list = motor_execution_runs
        task_type = "motor_execution"
    else:
        runs_list = motor_imagery_runs
        task_type = "motor_imagery"
    
    # Seleccionar un run aleatorio
    selected_run = random.choice(runs_list)
    
    # Determinar el paradigma (izquierda/derecha o manos/pies)
    if selected_run in [3, 4, 7, 8, 11, 12]:
        paradigm = "left_right_hand"
    else:  # runs 5, 6, 9, 10, 13, 14
        paradigm = "hands_feet"
    
    logger.info(f"Seleccionado sujeto: {subject}, run: {selected_run}")
    logger.info(f"Tipo de tarea: {task_type}, paradigma: {paradigm}")
    
    # Cargar los datos usando la función de MNE
    raw_files = eegbci.load_data(subject, [selected_run])
    
    if not raw_files:
        raise ValueError(f"No se encontraron archivos para el sujeto {subject}, run {selected_run}")
    
    # Leer y concatenar los archivos EDF
    raws = [read_raw_edf(f, preload=True) for f in raw_files]
    raw_data = concatenate_raws(raws)
    
    # Estandarizar nombres de canales al sistema internacional 10-20
    eegbci.standardize(raw_data)
    
    # Configurar montaje EEG
    montage = mne.channels.make_standard_montage('standard_1005')
    raw_data.set_montage(montage)
    
    # Guardar metadatos del sujeto
    raw_data.info['subject_info'] = {'his_id': str(subject)}
    
    # Guardar metadatos adicionales como atributo
    metadata = {
        'subject': subject,
        'task_type': task_type,
        'paradigm': paradigm,
        'run': selected_run
    }
    
    raw_data.metadata = metadata
    
    return raw_data, subject, selected_run, task_type, paradigm

# Función para preprocesar datos
def preprocess_data(raw_data, low_cutoff=4, high_cutoff=45, apply_notch=True, tmin=-1.0, tmax=4.0):
    """
    Aplica preprocesamiento a los datos EEG crudos.
    """
    # Crear copia para no modificar los datos originales
    filter_data = raw_data.copy()
    
    # Aplicar filtro pasa banda
    logger.info(f"Aplicando filtro pasa banda ({low_cutoff}-{high_cutoff} Hz)...")
    filter_data.filter(low_cutoff, high_cutoff, fir_design='firwin')
    
    # Aplicar filtro notch si es necesario
    if apply_notch:
        logger.info("Aplicando filtro notch a 60Hz...")
        filter_data.notch_filter(freqs=[60], fir_design='firwin')
    
    # Extraer eventos de las anotaciones
    events, event_id = mne.events_from_annotations(filter_data)
    
    # Mapear IDs de eventos a nombres más descriptivos
    metadata = getattr(filter_data, 'metadata', {})
    paradigm = metadata.get('paradigm', '')
    
    if paradigm == 'left_right_hand':
        new_event_id = {
            'rest': event_id.get('T0', 0),
            'left_hand': event_id.get('T1', 0),
            'right_hand': event_id.get('T2', 0)
        }
    else:  # hands_feet
        new_event_id = {
            'rest': event_id.get('T0', 0),
            'both_hands': event_id.get('T1', 0),
            'both_feet': event_id.get('T2', 0)
        }
    
    # Eliminar eventos con valor 0 (no encontrados)
    new_event_id = {k: v for k, v in new_event_id.items() if v != 0}
    
    logger.info(f"Mapeo de eventos: {new_event_id}")
    
    # Crear épocas
    epochs = mne.Epochs(
        filter_data,
        events,
        event_id=new_event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=(None, 0),
        preload=True
    )
    
    logger.info(f"Creadas {len(epochs)} épocas con {len(epochs.ch_names)} canales")
    
    # Extraer características para ML
    X = epochs.get_data()  # Forma: (n_epochs, n_channels, n_times)
    y = epochs.events[:, -1]  # Etiquetas
    
    # Reshape para ML (aplanar características)
    n_epochs, n_channels, n_times = X.shape
    X_flat = X.reshape(n_epochs, n_channels * n_times)
    
    logger.info(f"Datos extraídos: X shape {X_flat.shape}, y shape {y.shape}")
    
    return X_flat, y, epochs, new_event_id

# Función para cargar múltiples sujetos aleatorios
def load_random_subjects(num_subjects=6, random_seed=None):
    """
    Carga datos EEG de múltiples sujetos aleatorios.
    
    Args:
        num_subjects (int): Número de sujetos a cargar
        random_seed (int): Semilla para reproducibilidad
        
    Returns:
        list: Lista con información de EEG
    """
    if random_seed is not None:
        random.seed(random_seed)
        np.random.seed(random_seed)
    
    eeg_data = []
    selected_subjects = []
    
    print(f"Cargando {num_subjects} EEGs aleatorios...\n")
    
    for i in range(num_subjects):
        print(f"EEG #{i+1}:")
        
        # Cargar datos EEG, excluyendo sujetos ya seleccionados
        # Intentar hasta conseguir un sujeto válido
        max_attempts = 10
        for attempt in range(max_attempts):
            try:
                raw_data, subject, run, task_type, paradigm = load_random_subject(exclude_subjects=selected_subjects)
                selected_subjects.append(subject)
                break
            except Exception as e:
                if attempt == max_attempts - 1:
                    raise ValueError(f"No se pudo cargar un sujeto válido después de {max_attempts} intentos") from e
                print(f"Error al cargar el sujeto, reintentando ({attempt+1}/{max_attempts})...")
                continue
        
        # Preprocesar datos
        X, y, epochs, event_id = preprocess_data(
            raw_data, 
            PREPROCESSING_PARAMS['low_cutoff'],
            PREPROCESSING_PARAMS['high_cutoff'],
            PREPROCESSING_PARAMS['apply_notch'],
            PREPROCESSING_PARAMS['tmin'],
            PREPROCESSING_PARAMS['tmax']
        )
        
        # Guardar información relevante
        eeg_info = {
            'subject': subject,
            'run': run,
            'task_type': task_type,
            'paradigm': paradigm,
            'X': X,
            'y': y,
            'epochs': epochs,
            'event_id': event_id,
            'class_counts': {k: np.sum(y == v) for k, v in event_id.items()}
        }
        
        eeg_data.append(eeg_info)
        
        # Resumen
        print(f"  Sujeto: {subject}, Tarea: {task_type}, Paradigma: {paradigm}")
        print(f"  Forma de datos: {X.shape}")
        print(f"  Clases: {list(event_id.keys())}")
        print(f"  Distribución de clases: {eeg_info['class_counts']}")
        print()
    
    print(f"Cargados {len(eeg_data)} EEGs exitosamente.")
    return eeg_data

# Función para guardar metadatos de los sujetos
def save_metadata(eeg_data, filename='eeg_metadata.csv'):
    """Guarda los metadatos de los sujetos en un archivo CSV"""
    metadata = []
    for info in eeg_data:
        meta = {
            'subject': info['subject'],
            'run': info['run'],
            'task_type': info['task_type'],
            'paradigm': info['paradigm'],
            'num_samples': info['X'].shape[0],
            'num_features': info['X'].shape[1],
            'classes': ','.join(info['event_id'].keys()),
            'class_distribution': str(info['class_counts'])
        }
        metadata.append(meta)
    
    df = pd.DataFrame(metadata)
    df.to_csv(filename, index=False)
    logger.info(f"Metadatos guardados en {filename}")
    return df

# Función para normalizar etiquetas entre diferentes paradigmas
def normalize_labels(eeg_data):
    """
    Normaliza las etiquetas para garantizar compatibilidad entre paradigmas
    1 -> rest, 2 -> clase1 (left_hand/both_hands), 3 -> clase2 (right_hand/both_feet)
    """
    for info in eeg_data:
        mapping = {}
        event_id = info['event_id']
        
        # Asignar nuevos valores según el paradigma
        if 'rest' in event_id:
            mapping[event_id['rest']] = 1
        
        if 'left_hand' in event_id:
            mapping[event_id['left_hand']] = 2
        elif 'both_hands' in event_id:
            mapping[event_id['both_hands']] = 2
            
        if 'right_hand' in event_id:
            mapping[event_id['right_hand']] = 3
        elif 'both_feet' in event_id:
            mapping[event_id['both_feet']] = 3
        
        # Aplicar mapeo a las etiquetas
        y_original = info['y'].copy()
        for old_label, new_label in mapping.items():
            info['y'][info['y'] == old_label] = new_label
        
        # Actualizar metadata
        info['original_event_id'] = info['event_id'].copy()
        info['event_id'] = {'rest': 1, 'clase1': 2, 'clase2': 3}
        info['class_counts'] = {k: np.sum(info['y'] == v) for k, v in info['event_id'].items()}
        
    return eeg_data

# Si se ejecuta como script principal
if __name__ == "__main__":
    # Configuración del experimento
    NUM_EEGS = 6
    RANDOM_SEED = 42
    
    # Configurar semilla aleatoria
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    
    # Cargar sujetos aleatorios
    eeg_data = load_random_subjects(num_subjects=NUM_EEGS, random_seed=RANDOM_SEED)
    
    # Normalizar etiquetas para consistencia
    eeg_data = normalize_labels(eeg_data)
    
    # Guardar metadatos
    metadata_df = save_metadata(eeg_data)
    print("\nResumen de datos EEG:")
    print(metadata_df)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
from datetime import datetime
from joblib import dump
from sklearn.base import BaseEstimator, TransformerMixin

# Librerías para EEG
import mne
from mne.decoding import CSP

# Procesamiento de señales
from scipy import signal

# Preprocesamiento y ML
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# Configuración
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('eeg_pipeline')

# Ignorar warnings
warnings.filterwarnings('ignore')

# Directorio para modelos
MODELS_DIR = './models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Configuración del experimento
CV_FOLDS = 5
RANDOM_SEED = 42

# Clase para extraer características basadas en CSP
class CSPTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=4, reg=None, log=True, norm_trace=False):
        self.n_components = n_components
        self.reg = reg
        self.log = log
        self.norm_trace = norm_trace
        self.csp = CSP(n_components=n_components, reg=reg, log=log, norm_trace=norm_trace)
        
    def fit(self, X, y):
        # Reshape para CSP (n_trials, n_channels, n_times)
        n_trials, n_features = X.shape
        n_channels = 64  # Número de canales EEG
        n_times = n_features // n_channels
        X_reshaped = X.reshape(n_trials, n_channels, n_times)
        
        self.csp.fit(X_reshaped, y)
        return self
    
    def transform(self, X):
        # Reshape para CSP (n_trials, n_channels, n_times)
        n_trials, n_features = X.shape
        n_channels = 64  # Número de canales EEG
        n_times = n_features // n_channels
        X_reshaped = X.reshape(n_trials, n_channels, n_times)
        
        return self.csp.transform(X_reshaped)

# Clase para extraer características frecuenciales usando scipy
class FrequencyBandsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, sfreq=160, bands=None):
        self.sfreq = sfreq
        
        # Definir bandas de frecuencia si no se proporcionan
        if bands is None:
            self.bands = {
                'delta': (1, 4),
                'theta': (4, 8),
                'alpha_low': (8, 10),
                'alpha_high': (10, 13),
                'beta_low': (13, 16),
                'beta_mid': (16, 20),
                'beta_high': (20, 30),
                'gamma_low': (30, 45)
            }
        else:
            self.bands = bands
            
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Reshape a (n_trials, n_channels, n_times)
        n_trials, n_features = X.shape
        n_channels = 64
        n_times = n_features // n_channels
        X_reshaped = X.reshape(n_trials, n_channels, n_times)
        
        # Calcular PSD para cada trial usando scipy
        features = []
        
        for trial in X_reshaped:
            # Para cada canal, calcular PSD usando scipy
            trial_features = []
            
            for channel in trial:
                # Calcular PSD
                freqs, psd = signal.welch(channel, fs=self.sfreq, nperseg=256, nfft=1024)
                
                # Extraer características de bandas de frecuencia
                band_features = []
                for fmin, fmax in self.bands.values():
                    # Encontrar índices de frecuencia para la banda actual
                    idx_band = np.logical_and(freqs >= fmin, freqs <= fmax)
                    
                    # Extraer características de esta banda
                    if np.any(idx_band):
                        band_psd = psd[idx_band]
                        band_features.extend([
                            np.mean(band_psd),       # Potencia media
                            np.std(band_psd),        # Desviación estándar
                            np.max(band_psd),        # Potencia máxima
                            np.sum(band_psd)         # Potencia total
                        ])
                    else:
                        band_features.extend([0, 0, 0, 0])
                
                # Características temporales adicionales
                temporal_features = [
                    np.std(channel),           # Desviación estándar
                    np.max(np.abs(channel)),   # Amplitud máxima
                    np.mean(np.abs(channel)),  # Amplitud media
                    np.percentile(channel, 75) - np.percentile(channel, 25)  # IQR
                ]
                
                trial_features.extend(band_features + temporal_features)
            
            features.append(trial_features)
        
        return np.array(features)

# Función para crear pipeline avanzado
def create_advanced_pipeline(config='csp_svm', csp_components=6):
    """
    Crea un pipeline avanzado para clasificación EEG.
    
    Args:
        config (str): Configuración del pipeline
        csp_components (int): Número de componentes CSP
        
    Returns:
        Pipeline: Pipeline de scikit-learn configurado
    """
    # Estrategia de validación cruzada
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    
    if config == 'csp_svm':
        # Pipeline con CSP y SVM optimizado
        return Pipeline([
            ('scaler', RobustScaler()),
            ('csp', CSPTransformer(n_components=csp_components)),
            ('classifier', GridSearchCV(
                SVC(probability=True, class_weight='balanced'),
                param_grid={
                    'C': [1, 10, 100],
                    'gamma': ['scale', 'auto', 0.01],
                    'kernel': ['rbf']
                },
                cv=cv, scoring='accuracy', n_jobs=-1
            ))
        ])
    
    elif config == 'freq_rf':
        # Pipeline con características de frecuencia y Random Forest
        return Pipeline([
            ('scaler', StandardScaler()),
            ('freq_features', FrequencyBandsTransformer()),
            ('selector', SelectKBest(f_classif, k=50)),
            ('classifier', RandomForestClassifier(n_estimators=500, max_depth=None, 
                                                min_samples_split=2, bootstrap=True,
                                                class_weight='balanced', random_state=RANDOM_SEED,
                                                n_jobs=-1))
        ])
    
    elif config == 'csp_freq_rf':
        # Pipeline combinando CSP y características de frecuencia
        return Pipeline([
            ('scaler', StandardScaler()),
            ('csp', CSPTransformer(n_components=csp_components)),
            ('freq_features', FrequencyBandsTransformer()),
            ('pca', PCA(n_components=30)),
            ('classifier', RandomForestClassifier(n_estimators=300, max_depth=None,
                                               class_weight='balanced', random_state=RANDOM_SEED,
                                               n_jobs=-1))
        ])
    
    elif config == 'pca_mlp':
        # Pipeline con PCA y MLP con early stopping
        return Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=50)),
            ('classifier', MLPClassifier(hidden_layer_sizes=(100, 50), activation='relu',
                                        solver='adam', alpha=0.0001, batch_size='auto',
                                        learning_rate='adaptive', max_iter=300,
                                        early_stopping=True, validation_fraction=0.2,
                                        random_state=RANDOM_SEED))
        ])
    
    else:
        raise ValueError(f"Configuración desconocida: {config}")

# Función para comparar pipelines
def compare_pipelines(eeg_data, configs=None):
    """
    Compara diferentes configuraciones de pipelines.
    
    Args:
        eeg_data (list): Lista de información EEG
        configs (list): Lista de configuraciones de pipeline a probar
        
    Returns:
        dict: Resultados de la comparación
    """
    if configs is None:
        configs = ['csp_svm', 'freq_rf', 'csp_freq_rf', 'pca_mlp']
    
    results = {}
    
    # Preparar datos combinados
    X_combined = np.vstack([info['X'] for info in eeg_data])
    y_combined = np.concatenate([info['y'] for info in eeg_data])
    
    # Normalizar etiquetas si es necesario
    unique_labels = np.unique(y_combined)
    if len(unique_labels) > 3:  # Si hay más de 3 clases diferentes
        print("Encontradas más de 3 clases distintas. Normalizando etiquetas...")
        label_map = {}
        for i, label in enumerate(unique_labels):
            label_map[label] = i + 1
        
        # Aplicar mapeo
        for info in eeg_data:
            for old_label, new_label in label_map.items():
                info['y'][info['y'] == old_label] = new_label
        
        # Actualizar datos combinados
        y_combined = np.concatenate([info['y'] for info in eeg_data])
    
    print(f"Datos combinados: X shape {X_combined.shape}, y shape {y_combined.shape}")
    print(f"Clases: {np.unique(y_combined)}")
    print(f"Distribución de clases: {np.bincount(y_combined.astype(int))}")
    
    print("\nComparando configuraciones de pipelines con validación cruzada...")
    
    for config in configs:
        print(f"\nEvaluando pipeline: '{config}'")
        pipeline = create_advanced_pipeline(config)
        
        # Configurar validación cruzada
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
        
        # Métricas a evaluar
        accuracy_scores = []
        f1_scores = []
        training_times = []
        prediction_times = []
        
        # Validación cruzada
        fold = 1
        for train_idx, test_idx in skf.split(X_combined, y_combined):
            print(f"  Fold {fold}/{CV_FOLDS}...")
            X_train, X_test = X_combined[train_idx], X_combined[test_idx]
            y_train, y_test = y_combined[train_idx], y_combined[test_idx]
            
            # Entrenar
            start_time = datetime.now()
            pipeline.fit(X_train, y_train)
            train_time = (datetime.now() - start_time).total_seconds()
            
            # Predecir
            start_time = datetime.now()
            y_pred = pipeline.predict(X_test)
            predict_time = (datetime.now() - start_time).total_seconds()
            
            # Calcular métricas
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='weighted')
            
            # Guardar resultados
            accuracy_scores.append(acc)
            f1_scores.append(f1)
            training_times.append(train_time)
            prediction_times.append(predict_time)
            
            print(f"    Accuracy: {acc:.4f}, F1: {f1:.4f}, Time: {train_time:.2f}s")
            fold += 1
        
        # Calcular promedios y desviaciones
        scores = {}
        scores['accuracy'] = np.mean(accuracy_scores)
        scores['accuracy_std'] = np.std(accuracy_scores)
        scores['f1_weighted'] = np.mean(f1_scores)
        scores['f1_weighted_std'] = np.std(f1_scores)
        scores['training_time'] = np.mean(training_times)
        scores['prediction_time'] = np.mean(prediction_times)
        
        # Mostrar resultados finales
        print(f"  Accuracy CV: {scores['accuracy']:.4f} ± {scores['accuracy_std']:.4f}")
        print(f"  F1 Score CV: {scores['f1_weighted']:.4f} ± {scores['f1_weighted_std']:.4f}")
        print(f"  Tiempo promedio entrenamiento: {scores['training_time']:.2f}s")
        
        # Guardar resultados
        results[config] = scores
    
    # Identificar mejor configuración
    best_config = max(results, key=lambda k: results[k]['accuracy'])
    print(f"\nMejor configuración: '{best_config}' con accuracy {results[best_config]['accuracy']:.4f}")
    
    return {
        'results': results,
        'best_config': best_config
    }

def hold_one_out_experiment(eeg_data, pipeline_config='csp_svm'):
    """
    Experimento hold-one-out con pipeline avanzado.
    
    Args:
        eeg_data (list): Lista de información EEG
        pipeline_config (str): Configuración del pipeline a usar
        
    Returns:
        dict: Resultados del experimento
    """
    n_eegs = len(eeg_data)
    results = []
    
    print(f"Iniciando experimento hold-one-out con pipeline '{pipeline_config}'\n")
    
    for i in range(n_eegs):
        print(f"Iteración {i+1}/{n_eegs} - Excluyendo Sujeto {eeg_data[i]['subject']}")
        
        # Separar datos de test y entrenamiento
        test_data = eeg_data[i]
        train_data = [eeg_data[j] for j in range(n_eegs) if j != i]
        
        # Verificar compatibilidad de clases
        test_classes = set(test_data['event_id'].keys())
        all_compatible = True
        
        for train_item in train_data:
            train_classes = set(train_item['event_id'].keys())
            if train_classes != test_classes:
                all_compatible = False
                break
        
        if not all_compatible:
            print("  ⚠️ Advertencia: Las clases en los datos de entrenamiento no coinciden con las clases de test")
            print("  ⚠️ Normalizando etiquetas para garantizar compatibilidad")
            
            # Normalizar etiquetas para asegurar compatibilidad
            # 1 -> rest, 2 -> clase1 (left_hand/both_hands), 3 -> clase2 (right_hand/both_feet)
            
            # Mapeo para test_data
            test_mapping = {}
            for idx, key in enumerate(['rest', 
                                      'left_hand' if 'left_hand' in test_data['event_id'] else 'both_hands',
                                      'right_hand' if 'right_hand' in test_data['event_id'] else 'both_feet']):
                if key in test_data['event_id']:
                    test_mapping[test_data['event_id'][key]] = idx + 1
            
            # Aplicar mapeo a datos de test
            y_test_original = test_data['y'].copy()
            for old_label, new_label in test_mapping.items():
                test_data['y'][test_data['y'] == old_label] = new_label
            
            # Aplicar mapeo a datos de entrenamiento
            for train_item in train_data:
                train_mapping = {}
                for idx, key in enumerate(['rest', 
                                          'left_hand' if 'left_hand' in train_item['event_id'] else 'both_hands',
                                          'right_hand' if 'right_hand' in train_item['event_id'] else 'both_feet']):
                    if key in train_item['event_id']:
                        train_mapping[train_item['event_id'][key]] = idx + 1
                
                # Aplicar mapeo
                for old_label, new_label in train_mapping.items():
                    train_item['y'][train_item['y'] == old_label] = new_label
        
        # Combinar datos de entrenamiento
        X_train_combined = np.vstack([item['X'] for item in train_data])
        y_train_combined = np.concatenate([item['y'] for item in train_data])
        
        # Datos de test
        X_test = test_data['X']
        y_test = test_data['y']
        
        print(f"  Datos de entrenamiento: {X_train_combined.shape}, Datos de test: {X_test.shape}")
        print(f"  Clases en train: {np.unique(y_train_combined)}, Clases en test: {np.unique(y_test)}")
        
        # Crear y entrenar pipeline avanzado
        pipeline = create_advanced_pipeline(pipeline_config)
        
        print("  Entrenando modelo...")
        start_time = datetime.now()
        pipeline.fit(X_train_combined, y_train_combined)
        train_time = (datetime.now() - start_time).total_seconds()
        
        # Predecir en datos de test
        print("  Evaluando en datos de test...")
        start_time = datetime.now()
        y_pred = pipeline.predict(X_test)
        predict_time = (datetime.now() - start_time).total_seconds()
        
        # Calcular métricas
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        cm = confusion_matrix(y_test, y_pred)
        
        # Mapear IDs numéricos a nombres de clases
        if all_compatible:
            id_to_class = {v: k for k, v in test_data['event_id'].items()}
        else:
            # Usar mapeo genérico si se normalizaron las etiquetas
            id_to_class = {1: 'rest', 2: 'clase1', 3: 'clase2'}
        
        class_names = [id_to_class.get(c, f"Clase {c}") for c in sorted(np.unique(y_test))]
        
        # Generar reporte
        report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
        
        # Guardar resultados de esta iteración
        iter_results = {
            'subject': test_data['subject'],
            'paradigm': test_data['paradigm'],
            'task_type': test_data['task_type'],
            'accuracy': acc,
            'f1_score': f1,
            'train_time': train_time,
            'predict_time': predict_time,
            'confusion_matrix': cm,
            'classification_report': report,
            'y_true': y_test,
            'y_pred': y_pred,
            'class_mapping': id_to_class,
            'pipeline': pipeline  # Guardar el pipeline entrenado
        }
        
        results.append(iter_results)
        
        print(f"  Resultados: Accuracy = {acc:.4f}, F1 = {f1:.4f}")
        print(f"  Tiempo: Entrenamiento = {train_time:.2f}s, Predicción = {predict_time:.2f}s\n")
    
    # Calcular métricas promedio
    avg_accuracy = np.mean([r['accuracy'] for r in results])
    avg_f1 = np.mean([r['f1_score'] for r in results])
    
    print(f"Resultados finales del experimento:")
    print(f"  Promedio Accuracy: {avg_accuracy:.4f}")
    print(f"  Promedio F1 Score: {avg_f1:.4f}")
    
    return {
        'iterations': results,
        'avg_accuracy': avg_accuracy,
        'avg_f1': avg_f1,
        'pipeline_config': pipeline_config
    }

def plot_confusion_matrices(experiment_results):
    """
    Visualiza las matrices de confusión para cada iteración del experimento.
    """
    iterations = experiment_results['iterations']
    n_eegs = len(iterations)
    
    # Crear rejilla para gráficos
    n_cols = 3
    n_rows = (n_eegs + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
    if n_rows > 1:
        axes = axes.flatten()
    
    for i, result in enumerate(iterations):
        if n_eegs == 1:
            ax = axes
        else:
            ax = axes[i]
            
        cm = result['confusion_matrix']
        
        # Obtener nombres de clases
        class_mapping = result['class_mapping']
        class_names = [class_mapping.get(idx, f"Clase {idx}") for idx in sorted(class_mapping.keys())]
        
        # Crear heatmap
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, 
                  yticklabels=class_names, ax=ax, cbar=False)
        
        # Configurar título y etiquetas
        ax.set_title(f"Sujeto {result['subject']} - {result['paradigm']}\nAcc: {result['accuracy']:.3f}")
        ax.set_ylabel('Clase real')
        ax.set_xlabel('Clase predicha')
    
    # Ocultar ejes no utilizados
    if n_eegs < len(axes):
        for j in range(n_eegs, len(axes)):
            axes[j].axis('off')
    
    plt.tight_layout()
    plt.suptitle('Matrices de Confusión por Sujeto', y=1.02, fontsize=16)
    return fig

def plot_subject_performances(experiment_results):
    """
    Visualiza el rendimiento para cada sujeto en el experimento.
    """
    iterations = experiment_results['iterations']
    
    # Extraer datos para visualización
    subjects = [r['subject'] for r in iterations]
    accuracies = [r['accuracy'] for r in iterations]
    f1_scores = [r['f1_score'] for r in iterations]
    task_types = [r['task_type'] for r in iterations]
    paradigms = [r['paradigm'] for r in iterations]
    
    # Crear DataFrame
    df = pd.DataFrame({
        'Sujeto': subjects,
        'Accuracy': accuracies,
        'F1 Score': f1_scores,
        'Tipo de Tarea': task_types,
        'Paradigma': paradigms
    })
    
    # Ordenar por accuracy
    df_sorted = df.sort_values('Accuracy', ascending=False)
    
    # Crear gráfico
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Barras para accuracy y F1
    x = np.arange(len(df_sorted))
    width = 0.35
    
    ax.bar(x - width/2, df_sorted['Accuracy'], width, label='Accuracy', color='#3498db')
    ax.bar(x + width/2, df_sorted['F1 Score'], width, label='F1 Score', color='#2ecc71')
    
    # Configurar etiquetas de eje X
    labels = [f"S{s}\n({p[:1]}{'E' if t == 'motor_execution' else 'I'})" 
             for s, p, t in zip(df_sorted['Sujeto'], df_sorted['Paradigma'], df_sorted['Tipo de Tarea'])]
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    
    # Añadir etiquetas
    for i, acc in enumerate(df_sorted['Accuracy']):
        ax.text(i - width/2, acc + 0.01, f"{acc:.3f}", ha='center')
    
    for i, f1 in enumerate(df_sorted['F1 Score']):
        ax.text(i + width/2, f1 + 0.01, f"{f1:.3f}", ha='center')
    
    # Configurar gráfico
    ax.set_ylabel('Puntuación')
    ax.set_title('Rendimiento por Sujeto')
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Añadir línea para promedio
    avg_acc = experiment_results['avg_accuracy']
    ax.axhline(y=avg_acc, linestyle='--', color='#e74c3c', alpha=0.7)
    ax.text(len(df_sorted)-1, avg_acc + 0.02, f"Promedio: {avg_acc:.3f}", ha='right', color='#e74c3c')
    
    # Añadir leyenda para las abreviaturas
    legend_text = "Abreviaturas:\nL = left_right_hand, H = hands_feet\nE = motor_execution, I = motor_imagery"
    ax.text(0.02, -0.15, legend_text, transform=ax.transAxes, fontsize=9, bbox=dict(facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    return fig

def pipeline_comparison_chart(pipeline_results):
    """
    Crea un gráfico de barras comparando diferentes pipelines.
    """
    # Extraer resultados
    configs = list(pipeline_results['results'].keys())
    accuracies = [pipeline_results['results'][c]['accuracy'] for c in configs]
    f1_scores = [pipeline_results['results'][c]['f1_weighted'] for c in configs]
    acc_std = [pipeline_results['results'][c]['accuracy_std'] for c in configs]
    
    # Ordenar por accuracy
    sorted_indices = np.argsort(accuracies)[::-1]  # Orden descendente
    configs = [configs[i] for i in sorted_indices]
    accuracies = [accuracies[i] for i in sorted_indices]
    f1_scores = [f1_scores[i] for i in sorted_indices]
    acc_std = [acc_std[i] for i in sorted_indices]
    
    # Crear gráfico
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(configs))
    width = 0.35
    
    # Barras con error
    ax.bar(x - width/2, accuracies, width, yerr=acc_std, 
          label='Accuracy', color='#3498db', capsize=5)
    ax.bar(x + width/2, f1_scores, width, 
          label='F1 Score', color='#2ecc71')
    
    # Añadir etiquetas de valor
    for i, acc in enumerate(accuracies):
        ax.text(i - width/2, acc + acc_std[i] + 0.01, f"{acc:.3f}", ha='center')
    
    for i, f1 in enumerate(f1_scores):
        ax.text(i + width/2, f1 + 0.01, f"{f1:.3f}", ha='center')
    
    # Configurar gráfico
    ax.set_ylabel('Puntuación')
    ax.set_title('Comparación de Pipelines')
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.set_ylim(0, 1.1)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Resaltar el mejor pipeline
    best_config = pipeline_results['best_config']
    best_idx = configs.index(best_config)
    ax.get_xticklabels()[best_idx].set_color('red')
    ax.get_xticklabels()[best_idx].set_fontweight('bold')
    
    plt.tight_layout()
    return fig

def train_and_save_model(eeg_data, pipeline_config='csp_svm', model_name=None):
    """
    Entrena un modelo con todos los datos y lo guarda
    
    Args:
        eeg_data (list): Lista de información EEG
        pipeline_config (str): Configuración del pipeline
        model_name (str): Nombre base para el modelo
        
    Returns:
        tuple: (pipeline, model_info)
    """
    # Preparar datos combinados
    X_combined = np.vstack([info['X'] for info in eeg_data])
    y_combined = np.concatenate([info['y'] for info in eeg_data])
    
    # Crear pipeline
    pipeline = create_advanced_pipeline(pipeline_config)
    
    # Entrenar modelo
    print(f"Entrenando modelo final con pipeline '{pipeline_config}'...")
    start_time = datetime.now()
    pipeline.fit(X_combined, y_combined)
    train_time = (datetime.now() - start_time).total_seconds()
    
    print(f"Modelo entrenado en {train_time:.2f} segundos")
    
    # Evaluar modelo con validación cruzada
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    cv_scores = cross_val_score(pipeline, X_combined, y_combined, cv=cv, scoring='accuracy')
    
    print(f"CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    
    # Guardar modelo
    if model_name is None:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        model_name = f'eeg_model_{timestamp}'
    
    model_path = os.path.join(MODELS_DIR, f'{model_name}.joblib')
    dump(pipeline, model_path)
    
    # Preparar información del modelo
    subjects = [info['subject'] for info in eeg_data]
    paradigms = [info['paradigm'] for info in eeg_data]
    task_types = [info['task_type'] for info in eeg_data]
    
    model_info = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'pipeline_config': pipeline_config,
        'cv_accuracy': float(cv_scores.mean()),
        'cv_accuracy_std': float(cv_scores.std()),
        'subjects': subjects,
        'paradigms': paradigms,
        'task_types': task_types,
        'classes': sorted([str(c) for c in np.unique(y_combined)]),
        'class_distribution': {str(k): int(v) for k, v in zip(*np.unique(y_combined, return_counts=True))},
        'feature_shape': X_combined.shape,
        'training_time': train_time,
        'model_file': model_path
    }
    
    # Guardar información del modelo
    info_path = os.path.join(MODELS_DIR, f'{model_name}_info.json')
    with open(info_path, 'w') as f:
        json.dump(model_info, f, indent=4)
    
    print(f"Modelo guardado en: {model_path}")
    print(f"Información guardada en: {info_path}")
    
    return pipeline, model_info

if __name__ == "__main__":
    # Este script no debería ejecutarse directamente
    # sino importarse desde el notebook principal
    print("Este script está diseñado para importarse, no para ejecutarse directamente.")
    print("Ejemplo de uso en un notebook:")
    print("from preprocessing import load_random_subjects")
    print("from pipeline import compare_pipelines, hold_one_out_experiment, train_and_save_model")
    print()
    print("# Cargar datos")
    print("eeg_data = load_random_subjects(num_subjects=6)")
    print()
    print("# Comparar pipelines")
    print("results = compare_pipelines(eeg_data)")
    print()
    print("# Experimento hold-one-out")
    print("experiment = hold_one_out_experiment(eeg_data, results['best_config'])")
    print()
    print("# Entrenar y guardar modelo final")
    print("model, info = train_and_save_model(eeg_data, results['best_config'])")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
from joblib import load
from datetime import datetime
import mne
from mne.datasets import eegbci
from mne.io import read_raw_edf
from mne.viz import plot_raw

# Importar funciones de preprocessing
from preprocessing import preprocess_data

# Configuración
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('eeg_predict')

# Ignorar warnings
warnings.filterwarnings('ignore')

# Directorio para modelos
MODELS_DIR = './models'

def load_model(model_path=None):
    """
    Carga un modelo previamente entrenado.
    
    Args:
        model_path (str): Ruta al archivo del modelo. Si es None, busca el modelo más reciente.
        
    Returns:
        tuple: (pipeline, model_info)
    """
    if model_path is None:
        # Buscar el modelo más reciente
        model_files = [f for f in os.listdir(MODELS_DIR) if f.endswith('.joblib')]
        if not model_files:
            raise FileNotFoundError("No se encontraron modelos en el directorio de modelos")
        
        # Ordenar por fecha de modificación (más reciente primero)
        model_files.sort(key=lambda x: os.path.getmtime(os.path.join(MODELS_DIR, x)), reverse=True)
        model_path = os.path.join(MODELS_DIR, model_files[0])
        
        # Buscar archivo de información correspondiente
        info_path = model_path.replace('.joblib', '_info.json')
        if not os.path.exists(info_path):
            logger.warning(f"No se encontró archivo de información para el modelo {model_path}")
            info = None
        else:
            with open(info_path, 'r') as f:
                info = json.load(f)
    else:
        # Usar el modelo especificado
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"No se encontró el modelo en la ruta {model_path}")
        
        # Buscar archivo de información correspondiente
        info_path = model_path.replace('.joblib', '_info.json')
        if not os.path.exists(info_path):
            logger.warning(f"No se encontró archivo de información para el modelo {model_path}")
            info = None
        else:
            with open(info_path, 'r') as f:
                info = json.load(f)
    
    # Cargar el modelo
    logger.info(f"Cargando modelo desde {model_path}")
    pipeline = load(model_path)
    
    if info:
        logger.info(f"Modelo: {info.get('pipeline_config', 'Desconocido')}")
        logger.info(f"Accuracy CV: {info.get('cv_accuracy', 'Desconocido')}")
    
    return pipeline, info

def load_specific_subject(subject_id, run_id):
    """
    Carga los datos EEG de un sujeto específico.
    
    Args:
        subject_id (int): ID del sujeto
        run_id (int): ID del run
        
    Returns:
        tuple: (raw_data, task_type, paradigm)
    """
    logger.info(f"Cargando datos del sujeto {subject_id}, run {run_id}")
    
    # Cargar los datos usando la función de MNE
    raw_files = eegbci.load_data(subject_id, [run_id])
    
    if not raw_files:
        raise ValueError(f"No se encontraron archivos para el sujeto {subject_id}, run {run_id}")
    
    # Leer el archivo EDF
    raw_data = read_raw_edf(raw_files[0], preload=True)
    
    # Estandarizar nombres de canales al sistema internacional 10-20
    eegbci.standardize(raw_data)
    
    # Configurar montaje EEG
    montage = mne.channels.make_standard_montage('standard_1005')
    raw_data.set_montage(montage)
    
    # Determinar el tipo de tarea y paradigma
    if run_id in [3, 5, 7, 9, 11, 13]:
        task_type = "motor_execution"
    else:
        task_type = "motor_imagery"
    
    if run_id in [3, 4, 7, 8, 11, 12]:
        paradigm = "left_right_hand"
    else:  # runs 5, 6, 9, 10, 13, 14
        paradigm = "hands_feet"
    
    # Guardar metadatos
    metadata = {
        'subject': subject_id,
        'task_type': task_type,
        'paradigm': paradigm,
        'run': run_id
    }
    
    raw_data.metadata = metadata
    
    return raw_data, task_type, paradigm

def predict_eeg(raw_data, pipeline, show_raw=False, preprocessing_params=None):
    """
    Realiza predicciones sobre datos EEG.
    
    Args:
        raw_data (mne.io.Raw): Datos EEG crudos
        pipeline (sklearn.pipeline.Pipeline): Pipeline entrenado
        show_raw (bool): Si se muestra la visualización de datos crudos
        preprocessing_params (dict): Parámetros para el preprocesamiento
        
    Returns:
        dict: Resultados de la predicción
    """
    if preprocessing_params is None:
        preprocessing_params = {
            'low_cutoff': 4,
            'high_cutoff': 45,
            'apply_notch': True,
            'tmin': -1.0,
            'tmax': 4.0
        }
    
    # Visualizar datos crudos si se solicita
    if show_raw:
        fig = plot_raw(raw_data, title='Datos EEG crudos', show=False)
        plt.tight_layout()
        plt.show()
        
        results = {
            'X': X,
            'y_true': y,
            'y_pred': y_pred,
            'accuracy': accuracy,
            'f1_score': f1,
            'confusion_matrix': cm,
            'classification_report': report,
            'predict_time': predict_time,
            'class_mapping': id_to_class,
            'epochs': epochs
        }
    else:
        # Si no hay ground truth, solo devolver predicciones
        results = {
            'X': X,
            'y_pred': y_pred,
            'predict_time': predict_time,
            'epochs': epochs
        }
    
    return results

def visualize_predictions_over_time(predict_results):
    """
    Visualiza las predicciones sobre el tiempo.
    
    Args:
        predict_results (dict): Resultados de la predicción
    """
    # Obtener datos
    epochs = predict_results['epochs']
    y_pred = predict_results['y_pred']
    
    if 'y_true' in predict_results:
        y_true = predict_results['y_true']
        class_mapping = predict_results['class_mapping']
        
        # Mapear clases a nombres
        class_names = {v: k for k, v in class_mapping.items()}
        y_pred_names = [class_names.get(y, f"Clase {y}") for y in y_pred]
        y_true_names = [class_names.get(y, f"Clase {y}") for y in y_true]
        
        # Crear dataframe con predicciones por época
        df = pd.DataFrame({
            'Tiempo (s)': epochs.times,
            'Predicción': y_pred_names[0],
            'Clase real': y_true_names[0]
        })
        
        # Plot
        plt.figure(figsize=(12, 6))
        plt.plot(epochs.times, [1 if p == r else 0 for p, r in zip(y_pred, y_true)], 'g-', label='Predicción correcta')
        plt.axhline(y=0.5, linestyle='--', color='gray', alpha=0.5)
        plt.title('Predicciones a lo largo del tiempo')
        plt.xlabel('Tiempo (s)')
        plt.ylabel('Predicción correcta (1) / incorrecta (0)')
        plt.ylim(-0.1, 1.1)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        # Solo visualizar predicciones sin ground truth
        unique_predictions = np.unique(y_pred)
        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_predictions)))
        
        plt.figure(figsize=(12, 6))
        for i, pred_class in enumerate(unique_predictions):
            mask = y_pred == pred_class
            plt.scatter(np.arange(len(y_pred))[mask], np.ones(np.sum(mask))*i, 
                       label=f'Clase {pred_class}', color=colors[i], alpha=0.7)
        
        plt.title('Predicciones por época')
        plt.xlabel('Número de época')
        plt.ylabel('Clase predicha')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

def plot_feature_importance(pipeline, feature_names=None):
    """
    Visualiza la importancia de características para modelos que lo soporten.
    
    Args:
        pipeline (sklearn.pipeline.Pipeline): Pipeline entrenado
        feature_names (list): Nombres de las características
    """
    # Intentar obtener el estimador final
    if hasattr(pipeline, 'named_steps'):
        if 'classifier' in pipeline.named_steps:
            estimator = pipeline.named_steps['classifier']
            
            # Si es GridSearchCV, obtener el mejor estimador
            if hasattr(estimator, 'best_estimator_'):
                estimator = estimator.best_estimator_
            
            # Verificar si el modelo soporta importancia de características
            if hasattr(estimator, 'feature_importances_'):
                importances = estimator.feature_importances_
                
                # Si no se proporcionan nombres, usar índices
                if feature_names is None:
                    feature_names = [f'Característica {i}' for i in range(len(importances))]
                
                # Ordenar por importancia
                indices = np.argsort(importances)[::-1]
                
                # Tomar las 20 características más importantes
                top_n = 20
                indices = indices[:top_n]
                
                plt.figure(figsize=(10, 8))
                plt.title('Importancia de características')
                plt.barh(range(len(indices)), importances[indices], color='b', align='center')
                plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
                plt.xlabel('Importancia relativa')
                plt.tight_layout()
                plt.show()
            else:
                logger.warning("El modelo no soporta visualización de importancia de características")
        else:
            logger.warning("No se encontró un clasificador en el pipeline")
    else:
        logger.warning("El objeto proporcionado no es un pipeline válido")

def batch_predict(pipeline, data_directory, subject_ids=None, runs=None):
    """
    Realiza predicciones en lote para múltiples sujetos y runs.
    
    Args:
        pipeline (sklearn.pipeline.Pipeline): Pipeline entrenado
        data_directory (str): Directorio de datos
        subject_ids (list): Lista de IDs de sujetos, si es None, usa 1-5
        runs (list): Lista de runs, si es None, usa runs predeterminados
        
    Returns:
        dict: Resultados de las predicciones
    """
    if subject_ids is None:
        subject_ids = list(range(1, 6))  # Sujetos 1-5 por defecto
    
    if runs is None:
        # Usar un conjunto de runs motor_execution e imagery
        runs = [4, 8, 12]  # Imagery izquierda/derecha, ambas manos/pies
    
    results = {}
    
    for subject in subject_ids:
        subject_results = {}
        
        for run in runs:
            try:
                # Cargar datos
                raw_data, task_type, paradigm = load_specific_subject(subject, run)
                
                # Realizar predicción
                prediction = predict_eeg(raw_data, pipeline, show_raw=False)
                
                # Guardar resultados
                subject_results[f'run_{run}'] = {
                    'task_type': task_type,
                    'paradigm': paradigm,
                    'accuracy': prediction.get('accuracy', None),
                    'f1_score': prediction.get('f1_score', None),
                    'predict_time': prediction['predict_time']
                }
                
                logger.info(f"Sujeto {subject}, Run {run}: Predicción completada")
                
            except Exception as e:
                logger.error(f"Error en sujeto {subject}, run {run}: {str(e)}")
                subject_results[f'run_{run}'] = {'error': str(e)}
        
        results[f'subject_{subject}'] = subject_results
    
    # Generar resumen
    summary = {}
    for subject_key, subject_data in results.items():
        for run_key, run_data in subject_data.items():
            if 'accuracy' in run_data and run_data['accuracy'] is not None:
                if run_data['task_type'] not in summary:
                    summary[run_data['task_type']] = []
                
                summary[run_data['task_type']].append({
                    'subject': subject_key,
                    'run': run_key,
                    'paradigm': run_data['paradigm'],
                    'accuracy': run_data['accuracy'],
                    'f1_score': run_data['f1_score']
                })
    
    # Calcular promedios
    for task_type, task_data in summary.items():
        avg_acc = np.mean([item['accuracy'] for item in task_data])
        avg_f1 = np.mean([item['f1_score'] for item in task_data])
        
        logger.info(f"Promedio para {task_type}: Accuracy = {avg_acc:.4f}, F1 = {avg_f1:.4f}")
    
    return results, summary

if __name__ == "__main__":
    # Este script no debería ejecutarse directamente
    # sino importarse desde el notebook principal
    print("Este script está diseñado para importarse, no para ejecutarse directamente.")
    print("Ejemplo de uso en un notebook:")
    print("from predict import load_model, load_specific_subject, predict_eeg")
    print()
    print("# Cargar modelo")
    print("pipeline, model_info = load_model()")
    print()
    print("# Cargar datos de un sujeto específico")
    print("raw_data, task_type, paradigm = load_specific_subject(1, 4)")
    print()
    print("# Realizar predicción")
    print("results = predict_eeg(raw_data, pipeline, show_raw=True)")
    print()
    print("# Visualizar predicciones")
    print("from predict import visualize_predictions_over_time")
    print("visualize_predictions_over_time(results)")

    
    # Preprocesar datos
    logger.info("Preprocesando datos...")
    X, y, epochs, event_id = preprocess_data(
        raw_data,
        preprocessing_params['low_cutoff'],
        preprocessing_params['high_cutoff'],
        preprocessing_params['apply_notch'],
        preprocessing_params['tmin'],
        preprocessing_params['tmax']
    )
    
    # Realizar predicción
    logger.info("Realizando predicción...")
    start_time = datetime.now()
    y_pred = pipeline.predict(X)
    predict_time = (datetime.now() - start_time).total_seconds()
    
    # Si existe ground truth, calcular métricas
    if y is not None and len(y) > 0:
        from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
        accuracy = accuracy_score(y, y_pred)
        f1 = f1_score(y, y_pred, average='weighted')
        cm = confusion_matrix(y, y_pred)
        
        logger.info(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
        logger.info(f"Tiempo de predicción: {predict_time:.4f} segundos")
        
        # Mapear IDs numéricos a nombres de clases
        id_to_class = {v: k for k, v in event_id.items()}
        class_names = [id_to_class.get(c, f"Clase {c}") for c in sorted(np.unique(y))]
        
        # Generar reporte
        report = classification_report(y, y_pred, target_names=class_names, output_dict=True)
        
        # Visualizar matriz de confusión
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, 
                   yticklabels=class_names, cbar=False)
        plt.title(f"Matriz de Confusión\nAccuracy: {accuracy:.3f}, F1: {f1:.3f}")
        plt.ylabel('Clase real')
        plt.xlabel('Clase predicha')
        plt.tight_layout()
        plt.show()


In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import random
# import os
# import json
# import warnings
# from datetime import datetime
# from joblib import load, dump
# from tqdm import tqdm
# from sklearn.base import BaseEstimator, TransformerMixin

# # Librerías para EEG
# import mne
# from mne.datasets import eegbci
# from mne.io import read_raw_edf, concatenate_raws
# from mne.decoding import CSP
# from mne import Epochs, create_info

# # Procesamiento de señales
# from scipy import signal

# # Preprocesamiento y ML
# from sklearn.preprocessing import StandardScaler, RobustScaler
# from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
# from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
# from sklearn.pipeline import Pipeline
# from sklearn.decomposition import PCA
# from sklearn.feature_selection import SelectKBest, f_classif
# from sklearn.svm import SVC
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# from sklearn.neural_network import MLPClassifier

# # Configuración
# import logging
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
# logger = logging.getLogger('eeg_notebook')

# # Ignorar warnings
# warnings.filterwarnings('ignore')

# # Directorio para modelos
# MODELS_DIR = '../models'
# os.makedirs(MODELS_DIR, exist_ok=True)

# # Configuración del experimento
# NUM_EEGS = 6
# CV_FOLDS = 5
# RANDOM_SEED = 42
# random.seed(RANDOM_SEED)
# np.random.seed(RANDOM_SEED)

# # Parámetros de preprocesamiento
# PREPROCESSING_PARAMS = {
#     'low_cutoff': 4,      # Hz - Incluimos ondas theta (4-8 Hz)
#     'high_cutoff': 45,    # Hz - Incluimos hasta gama bajo (30-45 Hz)
#     'apply_notch': True,  # Filtro notch para ruido de línea eléctrica
#     'tmin': -1.0,         # Tiempo inicial para épocas (segundos)
#     'tmax': 4.0,          # Tiempo final para épocas (segundos)
#     'csp_components': 6   # Componentes CSP a utilizar
# }

# # Clase para extraer características basadas en CSP
# class CSPTransformer(BaseEstimator, TransformerMixin):
#     def __init__(self, n_components=4, reg=None, log=True, norm_trace=False):
#         self.n_components = n_components
#         self.reg = reg
#         self.log = log
#         self.norm_trace = norm_trace
#         self.csp = CSP(n_components=n_components, reg=reg, log=log, norm_trace=norm_trace)
        
#     def fit(self, X, y):
#         # Reshape para CSP (n_trials, n_channels, n_times)
#         n_trials, n_features = X.shape
#         n_channels = 64  # Número de canales EEG
#         n_times = n_features // n_channels
#         X_reshaped = X.reshape(n_trials, n_channels, n_times)
        
#         self.csp.fit(X_reshaped, y)
#         return self
    
#     def transform(self, X):
#         # Reshape para CSP (n_trials, n_channels, n_times)
#         n_trials, n_features = X.shape
#         n_channels = 64  # Número de canales EEG
#         n_times = n_features // n_channels
#         X_reshaped = X.reshape(n_trials, n_channels, n_times)
        
#         return self.csp.transform(X_reshaped)

# # Clase para extraer características frecuenciales usando scipy
# class FrequencyBandsTransformer(BaseEstimator, TransformerMixin):
#     def __init__(self, sfreq=160, bands=None):
#         self.sfreq = sfreq
        
#         # Definir bandas de frecuencia si no se proporcionan
#         if bands is None:
#             self.bands = {
#                 'delta': (1, 4),
#                 'theta': (4, 8),
#                 'alpha_low': (8, 10),
#                 'alpha_high': (10, 13),
#                 'beta_low': (13, 16),
#                 'beta_mid': (16, 20),
#                 'beta_high': (20, 30),
#                 'gamma_low': (30, 45)
#             }
#         else:
#             self.bands = bands
            
#     def fit(self, X, y=None):
#         return self
    
#     def transform(self, X):
#         # Reshape a (n_trials, n_channels, n_times)
#         n_trials, n_features = X.shape
#         n_channels = 64
#         n_times = n_features // n_channels
#         X_reshaped = X.reshape(n_trials, n_channels, n_times)
        
#         # Calcular PSD para cada trial usando scipy
#         features = []
        
#         for trial in X_reshaped:
#             # Para cada canal, calcular PSD usando scipy
#             trial_features = []
            
#             for channel in trial:
#                 # Calcular PSD
#                 freqs, psd = signal.welch(channel, fs=self.sfreq, nperseg=256, nfft=1024)
                
#                 # Extraer características de bandas de frecuencia
#                 band_features = []
#                 for fmin, fmax in self.bands.values():
#                     # Encontrar índices de frecuencia para la banda actual
#                     idx_band = np.logical_and(freqs >= fmin, freqs <= fmax)
                    
#                     # Extraer características de esta banda
#                     if np.any(idx_band):
#                         band_psd = psd[idx_band]
#                         band_features.extend([
#                             np.mean(band_psd),       # Potencia media
#                             np.std(band_psd),        # Desviación estándar
#                             np.max(band_psd),        # Potencia máxima
#                             np.sum(band_psd)         # Potencia total
#                         ])
#                     else:
#                         band_features.extend([0, 0, 0, 0])
                
#                 # Características temporales adicionales
#                 temporal_features = [
#                     np.std(channel),           # Desviación estándar
#                     np.max(np.abs(channel)),   # Amplitud máxima
#                     np.mean(np.abs(channel)),  # Amplitud media
#                     np.percentile(channel, 75) - np.percentile(channel, 25)  # IQR
#                 ]
                
#                 trial_features.extend(band_features + temporal_features)
            
#             features.append(trial_features)
        
#         return np.array(features)

# # Función para crear pipeline avanzado
# def create_advanced_pipeline(config='csp_svm'):
#     """
#     Crea un pipeline avanzado para clasificación EEG.
    
#     Args:
#         config (str): Configuración del pipeline
        
#     Returns:
#         Pipeline: Pipeline de scikit-learn configurado
#     """
#     # Estrategia de validación cruzada
#     cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    
#     if config == 'csp_svm':
#         # Pipeline con CSP y SVM optimizado
#         return Pipeline([
#             ('scaler', RobustScaler()),
#             ('csp', CSPTransformer(n_components=PREPROCESSING_PARAMS['csp_components'])),
#             ('classifier', GridSearchCV(
#                 SVC(probability=True, class_weight='balanced'),
#                 param_grid={
#                     'C': [1, 10, 100],
#                     'gamma': ['scale', 'auto', 0.01],
#                     'kernel': ['rbf']
#                 },
#                 cv=cv, scoring='accuracy', n_jobs=-1
#             ))
#         ])
    
#     elif config == 'freq_rf':
#         # Pipeline con características de frecuencia y Random Forest
#         return Pipeline([
#             ('scaler', StandardScaler()),
#             ('freq_features', FrequencyBandsTransformer()),
#             ('selector', SelectKBest(f_classif, k=50)),
#             ('classifier', RandomForestClassifier(n_estimators=500, max_depth=None, 
#                                                 min_samples_split=2, bootstrap=True,
#                                                 class_weight='balanced', random_state=RANDOM_SEED,
#                                                 n_jobs=-1))
#         ])
    
#     elif config == 'csp_freq_rf':
#         # Pipeline combinando CSP y características de frecuencia
#         return Pipeline([
#             ('scaler', StandardScaler()),
#             ('csp', CSPTransformer(n_components=PREPROCESSING_PARAMS['csp_components'])),
#             ('freq_features', FrequencyBandsTransformer()),
#             ('pca', PCA(n_components=30)),
#             ('classifier', RandomForestClassifier(n_estimators=300, max_depth=None,
#                                                class_weight='balanced', random_state=RANDOM_SEED,
#                                                n_jobs=-1))
#         ])
    
#     elif config == 'pca_mlp':
#         # Pipeline con PCA y MLP con early stopping
#         return Pipeline([
#             ('scaler', StandardScaler()),
#             ('pca', PCA(n_components=50)),
#             ('classifier', MLPClassifier(hidden_layer_sizes=(100, 50), activation='relu',
#                                         solver='adam', alpha=0.0001, batch_size='auto',
#                                         learning_rate='adaptive', max_iter=300,
#                                         early_stopping=True, validation_fraction=0.2,
#                                         random_state=RANDOM_SEED))
#         ])
    
#     else:
#         raise ValueError(f"Configuración desconocida: {config}")

# # Función para cargar un sujeto aleatorio
# def load_random_subject(exclude_subjects=None):
#     """
#     Carga datos EEG de un sujeto aleatorio, excluyendo los sujetos especificados.
    
#     Args:
#         exclude_subjects (list): Lista de IDs de sujetos a excluir
        
#     Returns:
#         tuple: (raw_data, subject_id, run_id, task_type, paradigm)
#     """
#     # Inicializar lista de exclusión si es None
#     if exclude_subjects is None:
#         exclude_subjects = []
    
#     # Determinar sujetos disponibles (1-109, excluyendo los ya seleccionados)
#     available_subjects = [s for s in range(1, 110) if s not in exclude_subjects]
    
#     if not available_subjects:
#         raise ValueError("No hay sujetos disponibles para seleccionar")
    
#     # Seleccionar un sujeto aleatorio
#     subject = random.choice(available_subjects)
    
#     # Definir los tipos de runs disponibles
#     motor_execution_runs = [3, 5, 7, 9, 11, 13]  # Izquierda/derecha o manos/pies
#     motor_imagery_runs = [4, 6, 8, 10, 12, 14]   # Izquierda/derecha o manos/pies
    
#     # Seleccionar aleatoriamente entre ejecución motora o imaginación motora
#     if random.choice([True, False]):
#         runs_list = motor_execution_runs
#         task_type = "motor_execution"
#     else:
#         runs_list = motor_imagery_runs
#         task_type = "motor_imagery"
    
#     # Seleccionar un run aleatorio
#     selected_run = random.choice(runs_list)
    
#     # Determinar el paradigma (izquierda/derecha o manos/pies)
#     if selected_run in [3, 4, 7, 8, 11, 12]:
#         paradigm = "left_right_hand"
#     else:  # runs 5, 6, 9, 10, 13, 14
#         paradigm = "hands_feet"
    
#     logger.info(f"Seleccionado sujeto: {subject}, run: {selected_run}")
#     logger.info(f"Tipo de tarea: {task_type}, paradigma: {paradigm}")
    
#     # Cargar los datos usando la función de MNE
#     raw_files = eegbci.load_data(subject, [selected_run])
    
#     if not raw_files:
#         raise ValueError(f"No se encontraron archivos para el sujeto {subject}, run {selected_run}")
    
#     # Leer y concatenar los archivos EDF
#     raws = [read_raw_edf(f, preload=True) for f in raw_files]
#     raw_data = concatenate_raws(raws)
    
#     # Estandarizar nombres de canales al sistema internacional 10-20
#     eegbci.standardize(raw_data)
    
#     # Configurar montaje EEG
#     montage = mne.channels.make_standard_montage('standard_1005')
#     raw_data.set_montage(montage)
    
#     # Guardar metadatos del sujeto
#     raw_data.info['subject_info'] = {'his_id': str(subject)}
    
#     # Guardar metadatos adicionales como atributo
#     metadata = {
#         'subject': subject,
#         'task_type': task_type,
#         'paradigm': paradigm,
#         'run': selected_run
#     }
    
#     raw_data.metadata = metadata
    
#     return raw_data, subject, selected_run, task_type, paradigm

# # Función para preprocesar datos
# def preprocess_data(raw_data, low_cutoff=4, high_cutoff=45, apply_notch=True, tmin=-1.0, tmax=4.0):
#     """
#     Aplica preprocesamiento a los datos EEG crudos.
#     """
#     # Crear copia para no modificar los datos originales
#     filter_data = raw_data.copy()
    
#     # Aplicar filtro pasa banda
#     logger.info(f"Aplicando filtro pasa banda ({low_cutoff}-{high_cutoff} Hz)...")
#     filter_data.filter(low_cutoff, high_cutoff, fir_design='firwin')
    
#     # Aplicar filtro notch si es necesario
#     if apply_notch:
#         logger.info("Aplicando filtro notch a 60Hz...")
#         filter_data.notch_filter(freqs=[60], fir_design='firwin')
    
#     # Extraer eventos de las anotaciones
#     events, event_id = mne.events_from_annotations(filter_data)
    
#     # Mapear IDs de eventos a nombres más descriptivos
#     metadata = getattr(filter_data, 'metadata', {})
#     paradigm = metadata.get('paradigm', '')
    
#     if paradigm == 'left_right_hand':
#         new_event_id = {
#             'rest': event_id.get('T0', 0),
#             'left_hand': event_id.get('T1', 0),
#             'right_hand': event_id.get('T2', 0)
#         }
#     else:  # hands_feet
#         new_event_id = {
#             'rest': event_id.get('T0', 0),
#             'both_hands': event_id.get('T1', 0),
#             'both_feet': event_id.get('T2', 0)
#         }
    
#     # Eliminar eventos con valor 0 (no encontrados)
#     new_event_id = {k: v for k, v in new_event_id.items() if v != 0}
    
#     logger.info(f"Mapeo de eventos: {new_event_id}")
    
#     # Crear épocas
#     epochs = mne.Epochs(
#         filter_data,
#         events,
#         event_id=new_event_id,
#         tmin=tmin,
#         tmax=tmax,
#         baseline=(None, 0),
#         preload=True
#     )
    
#     logger.info(f"Creadas {len(epochs)} épocas con {len(epochs.ch_names)} canales")
    
#     # Extraer características para ML
#     X = epochs.get_data()  # Forma: (n_epochs, n_channels, n_times)
#     y = epochs.events[:, -1]  # Etiquetas
    
#     # Reshape para ML (aplanar características)
#     n_epochs, n_channels, n_times = X.shape
#     X_flat = X.reshape(n_epochs, n_channels * n_times)
    
#     logger.info(f"Datos extraídos: X shape {X_flat.shape}, y shape {y.shape}")
    
#     return X_flat, y, epochs, new_event_id

# # Función para cargar múltiples sujetos aleatorios
# def load_random_subjects(num_subjects=6):
#     """
#     Carga datos EEG de múltiples sujetos aleatorios.
    
#     Args:
#         num_subjects (int): Número de sujetos a cargar
        
#     Returns:
#         list: Lista con información de EEG
#     """
#     eeg_data = []
#     selected_subjects = []
    
#     print(f"Cargando {num_subjects} EEGs aleatorios...\n")
    
#     for i in range(num_subjects):
#         print(f"EEG #{i+1}:")
        
#         # Cargar datos EEG, excluyendo sujetos ya seleccionados
#         # Intentar hasta conseguir un sujeto válido
#         max_attempts = 10
#         for attempt in range(max_attempts):
#             try:
#                 raw_data, subject, run, task_type, paradigm = load_random_subject(exclude_subjects=selected_subjects)
#                 selected_subjects.append(subject)
#                 break
#             except Exception as e:
#                 if attempt == max_attempts - 1:
#                     raise ValueError(f"No se pudo cargar un sujeto válido después de {max_attempts} intentos") from e
#                 print(f"Error al cargar el sujeto, reintentando ({attempt+1}/{max_attempts})...")
#                 continue
        
#         # Preprocesar datos
#         X, y, epochs, event_id = preprocess_data(
#             raw_data, 
#             PREPROCESSING_PARAMS['low_cutoff'],
#             PREPROCESSING_PARAMS['high_cutoff'],
#             PREPROCESSING_PARAMS['apply_notch'],
#             PREPROCESSING_PARAMS['tmin'],
#             PREPROCESSING_PARAMS['tmax']
#         )
        
#         # Guardar información relevante
#         eeg_info = {
#             'subject': subject,
#             'run': run,
#             'task_type': task_type,
#             'paradigm': paradigm,
#             'X': X,
#             'y': y,
#             'epochs': epochs,
#             'event_id': event_id,
#             'class_counts': {k: np.sum(y == v) for k, v in event_id.items()}
#         }
        
#         eeg_data.append(eeg_info)
        
#         # Resumen
#         print(f"  Sujeto: {subject}, Tarea: {task_type}, Paradigma: {paradigm}")
#         print(f"  Forma de datos: {X.shape}")
#         print(f"  Clases: {list(event_id.keys())}")
#         print(f"  Distribución de clases: {eeg_info['class_counts']}")
#         print()
    
#     print(f"Cargados {len(eeg_data)} EEGs exitosamente.")
#     return eeg_data

# # Función para comparar pipelines
# def compare_pipelines(eeg_data, configs=None):
#     """
#     Compara diferentes configuraciones de pipelines.
#     """
#     if configs is None:
#         configs = ['csp_svm', 'freq_rf', 'csp_freq_rf', 'pca_mlp']
    
#     results = {}
    
#     # Preparar datos combinados
#     X_combined = np.vstack([info['X'] for info in eeg_data])
#     y_combined = np.concatenate([info['y'] for info in eeg_data])
    
#     # Normalizar etiquetas si es necesario
#     unique_labels = np.unique(y_combined)
#     if len(unique_labels) > 3:  # Si hay más de 3 clases diferentes
#         print("Encontradas más de 3 clases distintas. Normalizando etiquetas...")
#         label_map = {}
#         for i, label in enumerate(unique_labels):
#             label_map[label] = i + 1
        
#         # Aplicar mapeo
#         for info in eeg_data:
#             for old_label, new_label in label_map.items():
#                 info['y'][info['y'] == old_label] = new_label
        
#         # Actualizar datos combinados
#         y_combined = np.concatenate([info['y'] for info in eeg_data])
    
#     print(f"Datos combinados: X shape {X_combined.shape}, y shape {y_combined.shape}")
#     print(f"Clases: {np.unique(y_combined)}")
#     print(f"Distribución de clases: {np.bincount(y_combined.astype(int))}")
    
#     print("\nComparando configuraciones de pipelines con validación cruzada...")
    
#     for config in configs:
#         print(f"\nEvaluando pipeline: '{config}'")
#         pipeline = create_advanced_pipeline(config)
        
#         # Configurar validación cruzada
#         skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
        
#         # Métricas a evaluar
#         accuracy_scores = []
#         f1_scores = []
#         training_times = []
#         prediction_times = []
        
#         # Validación cruzada
#         fold = 1
#         for train_idx, test_idx in skf.split(X_combined, y_combined):
#             print(f"  Fold {fold}/{CV_FOLDS}...")
#             X_train, X_test = X_combined[train_idx], X_combined[test_idx]
#             y_train, y_test = y_combined[train_idx], y_combined[test_idx]
            
#             # Entrenar
#             start_time = datetime.now()
#             pipeline.fit(X_train, y_train)
#             train_time = (datetime.now() - start_time).total_seconds()
            
#             # Predecir
#             start_time = datetime.now()
#             y_pred = pipeline.predict(X_test)
#             predict_time = (datetime.now() - start_time).total_seconds()
            
#             # Calcular métricas
#             acc = accuracy_score(y_test, y_pred)
#             f1 = f1_score(y_test, y_pred, average='weighted')
            
#             # Guardar resultados
#             accuracy_scores.append(acc)
#             f1_scores.append(f1)
#             training_times.append(train_time)
#             prediction_times.append(predict_time)
            
#             print(f"    Accuracy: {acc:.4f}, F1: {f1:.4f}, Time: {train_time:.2f}s")
#             fold += 1
        
#         # Calcular promedios y desviaciones
#         scores = {}
#         scores['accuracy'] = np.mean(accuracy_scores)
#         scores['accuracy_std'] = np.std(accuracy_scores)
#         scores['f1_weighted'] = np.mean(f1_scores)
#         scores['f1_weighted_std'] = np.std(f1_scores)
#         scores['training_time'] = np.mean(training_times)
#         scores['prediction_time'] = np.mean(prediction_times)
        
#         # Mostrar resultados finales
#         print(f"  Accuracy CV: {scores['accuracy']:.4f} ± {scores['accuracy_std']:.4f}")
#         print(f"  F1 Score CV: {scores['f1_weighted']:.4f} ± {scores['f1_weighted_std']:.4f}")
#         print(f"  Tiempo promedio entrenamiento: {scores['training_time']:.2f}s")
        
#         # Guardar resultados
#         results[config] = scores
    
#     # Identificar mejor configuración
#     best_config = max(results, key=lambda k: results[k]['accuracy'])
#     print(f"\nMejor configuración: '{best_config}' con accuracy {results[best_config]['accuracy']:.4f}")
    
#     return {
#         'results': results,
#         'best_config': best_config
#     }

# def hold_one_out_experiment(eeg_data, pipeline_config='csp_svm'):
#     """
#     Experimento hold-one-out con pipeline avanzado.
#     """
#     n_eegs = len(eeg_data)
#     results = []
    
#     print(f"Iniciando experimento hold-one-out con pipeline '{pipeline_config}'\n")
    
#     for i in range(n_eegs):
#         print(f"Iteración {i+1}/{n_eegs} - Excluyendo Sujeto {eeg_data[i]['subject']}")
        
#         # Separar datos de test y entrenamiento
#         test_data = eeg_data[i]
#         train_data = [eeg_data[j] for j in range(n_eegs) if j != i]
        
#         # Verificar compatibilidad de clases
#         test_classes = set(test_data['event_id'].keys())
#         all_compatible = True
        
#         for train_item in train_data:
#             train_classes = set(train_item['event_id'].keys())
#             if train_classes != test_classes:
#                 all_compatible = False
#                 break
        
#         if not all_compatible:
#             print("  ⚠️ Advertencia: Las clases en los datos de entrenamiento no coinciden con las clases de test")
#             print("  ⚠️ Normalizando etiquetas para garantizar compatibilidad")
            
#             # Normalizar etiquetas para asegurar compatibilidad
#             # 1 -> rest, 2 -> clase1 (left_hand/both_hands), 3 -> clase2 (right_hand/both_feet)
            
#             # Mapeo para test_data
#             test_mapping = {}
#             for idx, key in enumerate(['rest', 
#                                       'left_hand' if 'left_hand' in test_data['event_id'] else 'both_hands',
#                                       'right_hand' if 'right_hand' in test_data['event_id'] else 'both_feet']):
#                 if key in test_data['event_id']:
#                     test_mapping[test_data['event_id'][key]] = idx + 1
            
#             # Aplicar mapeo a datos de test
#             y_test_original = test_data['y'].copy()
#             for old_label, new_label in test_mapping.items():
#                 test_data['y'][test_data['y'] == old_label] = new_label
            
#             # Aplicar mapeo a datos de entrenamiento
#             for train_item in train_data:
#                 train_mapping = {}
#                 for idx, key in enumerate(['rest', 
#                                           'left_hand' if 'left_hand' in train_item['event_id'] else 'both_hands',
#                                           'right_hand' if 'right_hand' in train_item['event_id'] else 'both_feet']):
#                     if key in train_item['event_id']:
#                         train_mapping[train_item['event_id'][key]] = idx + 1
                
#                 # Aplicar mapeo
#                 for old_label, new_label in train_mapping.items():
#                     train_item['y'][train_item['y'] == old_label] = new_label
        
#         # Combinar datos de entrenamiento
#         X_train_combined = np.vstack([item['X'] for item in train_data])
#         y_train_combined = np.concatenate([item['y'] for item in train_data])
        
#         # Datos de test
#         X_test = test_data['X']
#         y_test = test_data['y']
        
#         print(f"  Datos de entrenamiento: {X_train_combined.shape}, Datos de test: {X_test.shape}")
#         print(f"  Clases en train: {np.unique(y_train_combined)}, Clases en test: {np.unique(y_test)}")
        
#         # Crear y entrenar pipeline avanzado
#         pipeline = create_advanced_pipeline(pipeline_config)
        
#         print("  Entrenando modelo...")
#         start_time = datetime.now()
#         pipeline.fit(X_train_combined, y_train_combined)
#         train_time = (datetime.now() - start_time).total_seconds()
        
#         # Predecir en datos de test
#         print("  Evaluando en datos de test...")
#         start_time = datetime.now()
#         y_pred = pipeline.predict(X_test)
#         predict_time = (datetime.now() - start_time).total_seconds()
        
#         # Calcular métricas
#         acc = accuracy_score(y_test, y_pred)
#         f1 = f1_score(y_test, y_pred, average='weighted')
#         cm = confusion_matrix(y_test, y_pred)
        
#         # Mapear IDs numéricos a nombres de clases
#         if all_compatible:
#             id_to_class = {v: k for k, v in test_data['event_id'].items()}
#         else:
#             # Usar mapeo genérico si se normalizaron las etiquetas
#             id_to_class = {1: 'rest', 2: 'clase1', 3: 'clase2'}
        
#         class_names = [id_to_class.get(c, f"Clase {c}") for c in sorted(np.unique(y_test))]
        
#         # Generar reporte
#         report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
        
#         # Guardar resultados de esta iteración
#         iter_results = {
#             'subject': test_data['subject'],
#             'paradigm': test_data['paradigm'],
#             'task_type': test_data['task_type'],
#             'accuracy': acc,
#             'f1_score': f1,
#             'train_time': train_time,
#             'predict_time': predict_time,
#             'confusion_matrix': cm,
#             'classification_report': report,
#             'y_true': y_test,
#             'y_pred': y_pred,
#             'class_mapping': id_to_class
#         }
        
#         results.append(iter_results)
        
#         print(f"  Resultados: Accuracy = {acc:.4f}, F1 = {f1:.4f}")
#         print(f"  Tiempo: Entrenamiento = {train_time:.2f}s, Predicción = {predict_time:.2f}s\n")
    
#     # Calcular métricas promedio
#     avg_accuracy = np.mean([r['accuracy'] for r in results])
#     avg_f1 = np.mean([r['f1_score'] for r in results])
    
#     print(f"Resultados finales del experimento:")
#     print(f"  Promedio Accuracy: {avg_accuracy:.4f}")
#     print(f"  Promedio F1 Score: {avg_f1:.4f}")
    
#     return {
#         'iterations': results,
#         'avg_accuracy': avg_accuracy,
#         'avg_f1': avg_f1,
#         'pipeline_config': pipeline_config
#     }

# def plot_confusion_matrices(experiment_results):
#     """
#     Visualiza las matrices de confusión para cada iteración del experimento.
#     """
#     iterations = experiment_results['iterations']
#     n_eegs = len(iterations)
    
#     # Crear rejilla para gráficos
#     n_cols = 3
#     n_rows = (n_eegs + n_cols - 1) // n_cols
#     fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
#     if n_rows > 1:
#         axes = axes.flatten()
    
#     for i, result in enumerate(iterations):
#         if n_eegs == 1:
#             ax = axes
#         else:
#             ax = axes[i]
            
#         cm = result['confusion_matrix']
        
#         # Obtener nombres de clases
#         class_mapping = result['class_mapping']
#         class_names = [class_mapping.get(idx, f"Clase {idx}") for idx in sorted(class_mapping.keys())]
        
#         # Crear heatmap
#         sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, 
#                    yticklabels=class_names, ax=ax, cbar=False)
        
#         # Configurar título y etiquetas
#         ax.set_title(f"Sujeto {result['subject']} - {result['paradigm']}\nAcc: {result['accuracy']:.3f}")
#         ax.set_ylabel('Clase real')
#         ax.set_xlabel('Clase predicha')
    
#     # Ocultar ejes no utilizados
#     if n_eegs < len(axes):
#         for j in range(n_eegs, len(axes)):
#             axes[j].axis('off')
    
#     plt.tight_layout()
#     plt.suptitle('Matrices de Confusión por Sujeto', y=1.02, fontsize=16)
#     plt.show()

# def plot_subject_performances(experiment_results):
#     """
#     Visualiza el rendimiento para cada sujeto en el experimento.
#     """
#     iterations = experiment_results['iterations']
    
#     # Extraer datos para visualización
#     subjects = [r['subject'] for r in iterations]
#     accuracies = [r['accuracy'] for r in iterations]
#     f1_scores = [r['f1_score'] for r in iterations]
#     task_types = [r['task_type'] for r in iterations]
#     paradigms = [r['paradigm'] for r in iterations]
    
#     # Crear DataFrame
#     df = pd.DataFrame({
#         'Sujeto': subjects,
#         'Accuracy': accuracies,
#         'F1 Score': f1_scores,
#         'Tipo de Tarea': task_types,
#         'Paradigma': paradigms
#     })
    
#     # Ordenar por accuracy
#     df_sorted = df.sort_values('Accuracy', ascending=False)
    
#     # Crear gráfico
#     fig, ax = plt.subplots(figsize=(12, 6))
    
#     # Barras para accuracy y F1
#     x = np.arange(len(df_sorted))
#     width = 0.35
    
#     ax.bar(x - width/2, df_sorted['Accuracy'], width, label='Accuracy', color='#3498db')
#     ax.bar(x + width/2, df_sorted['F1 Score'], width, label='F1 Score', color='#2ecc71')
    
#     # Configurar etiquetas de eje X
#     labels = [f"S{s}\n({p[:1]}{'E' if t == 'motor_execution' else 'I'})" 
#               for s, p, t in zip(df_sorted['Sujeto'], df_sorted['Paradigma'], df_sorted['Tipo de Tarea'])]
#     ax.set_xticks(x)
#     ax.set_xticklabels(labels)
    
#     # Añadir etiquetas
#     for i, acc in enumerate(df_sorted['Accuracy']):
#         ax.text(i - width/2, acc + 0.01, f"{acc:.3f}", ha='center')
    
#     for i, f1 in enumerate(df_sorted['F1 Score']):
#         ax.text(i + width/2, f1 + 0.01, f"{f1:.3f}", ha='center')
    
#     # Configurar gráfico
#     ax.set_ylabel('Puntuación')
#     ax.set_title('Rendimiento por Sujeto')
#     ax.set_ylim(0, 1.1)
#     ax.legend()
#     ax.grid(axis='y', linestyle='--', alpha=0.7)
    
#     # Añadir línea para promedio
#     avg_acc = experiment_results['avg_accuracy']
#     ax.axhline(y=avg_acc, linestyle='--', color='#e74c3c', alpha=0.7)
#     ax.text(len(df_sorted)-1, avg_acc + 0.02, f"Promedio: {avg_acc:.3f}", ha='right', color='#e74c3c')
    
#     # Añadir leyenda para las abreviaturas
#     legend_text = "Abreviaturas:\nL = left_right_hand, H = hands_feet\nE = motor_execution, I = motor_imagery"
#     ax.text(0.02, -0.15, legend_text, transform=ax.transAxes, fontsize=9, bbox=dict(facecolor='white', alpha=0.8))
    
#     plt.tight_layout()
#     plt.show()

# def pipeline_comparison_chart(pipeline_results):
#     """
#     Crea un gráfico de barras comparando diferentes pipelines.
#     """
#     # Extraer resultados
#     configs = list(pipeline_results['results'].keys())
#     accuracies = [pipeline_results['results'][c]['accuracy'] for c in configs]
#     f1_scores = [pipeline_results['results'][c]['f1_weighted'] for c in configs]
#     acc_std = [pipeline_results['results'][c]['accuracy_std'] for c in configs]
    
#     # Ordenar por accuracy
#     sorted_indices = np.argsort(accuracies)[::-1]  # Orden descendente
#     configs = [configs[i] for i in sorted_indices]
#     accuracies = [accuracies[i] for i in sorted_indices]
#     f1_scores = [f1_scores[i] for i in sorted_indices]
#     acc_std = [acc_std[i] for i in sorted_indices]
    
#     # Crear gráfico
#     fig, ax = plt.subplots(figsize=(10, 6))
    
#     x = np.arange(len(configs))
#     width = 0.35
    
#     # Barras con error
#     ax.bar(x - width/2, accuracies, width, yerr=acc_std, 
#            label='Accuracy', color='#3498db', capsize=5)
#     ax.bar(x + width/2, f1_scores, width, 
#            label='F1 Score', color='#2ecc71')
    
#     # Añadir etiquetas de valor
#     for i, acc in enumerate(accuracies):
#         ax.text(i - width/2, acc + acc_std[i] + 0.01, f"{acc:.3f}", ha='center')
    
#     for i, f1 in enumerate(f1_scores):
#         ax.text(i + width/2, f1 + 0.01, f"{f1:.3f}", ha='center')
    
#     # Configurar gráfico
#     ax.set_ylabel('Puntuación')
#     ax.set_title('Comparación de Pipelines')
#     ax.set_xticks(x)
#     ax.set_xticklabels(configs)
#     ax.set_ylim(0, 1.1)
#     ax.legend()
#     ax.grid(axis='y', linestyle='--', alpha=0.7)
    
#     # Resaltar el mejor pipeline
#     best_config = pipeline_results['best_config']
#     best_idx = configs.index(best_config)
#     ax.get_xticklabels()[best_idx].set_color('red')
#     ax.get_xticklabels()[best_idx].set_fontweight('bold')
    
#     plt.tight_layout()
#     plt.show()

# def main_random():
#     """
#     Función principal usando selección aleatoria de sujetos y experimentos.
#     """
#     print("===== Experimento Avanzado de Clasificación EEG con Selección Aleatoria =====\n")
    
#     try:
#         # Semilla aleatoria diferente para cada ejecución
#         random_seed = int(datetime.now().timestamp()) % 10000
#         random.seed(random_seed)
#         np.random.seed(random_seed)
#         print(f"Usando semilla aleatoria: {random_seed}")
        
#         # Cargar sujetos aleatorios
#         eeg_data = load_random_subjects(num_subjects=NUM_EEGS)
        
#         # Resumen de datos
#         eeg_summary = pd.DataFrame([
#             {
#                 'Subject': info['subject'],
#                 'Task Type': info['task_type'],
#                 'Paradigm': info['paradigm'],
#                 'Num Samples': info['X'].shape[0],
#                 'Num Features': info['X'].shape[1],
#                 'Classes': ', '.join(info['event_id'].keys())
#             } for info in eeg_data
#         ])
        
#         print("\nResumen de datos EEG:")
#         print(eeg_summary)
        
#         # Seleccionar configuraciones de pipeline aleatorias para comparar
#         all_configs = ['csp_svm', 'freq_rf', 'csp_freq_rf', 'pca_mlp']
#         num_configs = random.randint(2, len(all_configs))  # Al menos 2 configuraciones
#         configs_to_test = random.sample(all_configs, num_configs)
        
#         print(f"\nSeleccionadas {num_configs} configuraciones de pipeline para comparar: {configs_to_test}")
        
#         # Comparar pipelines
#         print("\n----- Comparación de Pipelines con Validación Cruzada -----")
#         pipeline_comparison = compare_pipelines(eeg_data, configs=configs_to_test)
        
#         # Visualizar comparación de pipelines
#         pipeline_comparison_chart(pipeline_comparison)
        
#         # Experimento hold-one-out con el mejor pipeline
#         best_config = pipeline_comparison['best_config']
#         print(f"\n----- Experimento Hold-One-Out con '{best_config}' -----")
#         results = hold_one_out_experiment(eeg_data, best_config)
        
#         # Visualizar matrices de confusión
#         plot_confusion_matrices(results)
        
#         # Visualizar rendimiento por sujeto
#         plot_subject_performances(results)
        
#         print("\n===== Experimento Aleatorio Completado =====")
#         print(f"Mejor pipeline: {best_config}")
#         print(f"Accuracy promedio: {results['avg_accuracy']:.4f}")
#         print(f"F1 Score promedio: {results['avg_f1']:.4f}")
        
#         # Guardar modelo y resultados
#         timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
#         save_path = os.path.join(MODELS_DIR, f'eeg_model_{timestamp}.joblib')
#         best_pipeline = results['iterations'][0]['pipeline']
#         dump(best_pipeline, save_path)
        
#         # Guardar información del modelo
#         info_path = os.path.join(MODELS_DIR, f'eeg_model_info_{timestamp}.json')
#         model_info = {
#             'timestamp': timestamp,
#             'random_seed': random_seed,
#             'pipeline_config': best_config,
#             'accuracy': float(results['avg_accuracy']),
#             'f1_score': float(results['avg_f1']),
#             'subjects': [int(info['subject']) for info in eeg_data],
#             'paradigms': [info['paradigm'] for info in eeg_data],
#             'task_types': [info['task_type'] for info in eeg_data],
#             'runs': [int(info['run']) for info in eeg_data]
#         }
        
#         with open(info_path, 'w') as f:
#             json.dump(model_info, f, indent=4)
            
#         print(f"\nModelo guardado en: {save_path}")
#         print(f"Información del modelo guardada en: {info_path}")
        
#     except Exception as e:
#         logger.error(f"Error en el experimento: {str(e)}")
#         raise e

# if __name__ == "__main__":
#     main_random()


2025-03-14 19:04:40,994 - eeg_notebook - INFO - Seleccionado sujeto: 61, run: 6
2025-03-14 19:04:40,997 - eeg_notebook - INFO - Tipo de tarea: motor_imagery, paradigma: hands_feet


===== Experimento Avanzado de Clasificación EEG con Selección Aleatoria =====

Usando semilla aleatoria: 9080
Cargando 6 EEGs aleatorios...

EEG #1:


Download complete in 07s (2.5 MB)
Extracting EDF parameters from /workspaces/42_vortex/data/raw/files/MNE-eegbci-data/files/eegmmidb/1.0.0/S061/S061R06.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...


2025-03-14 19:04:48,283 - eeg_notebook - INFO - Aplicando filtro pasa banda (4-45 Hz)...


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 4.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 3.00 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 265 samples (1.656 s)



...

Not setting metadata
30 matching events found
Setting baseline interval to [-1.0, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 30 events and 801 original time points ...
1 bad epochs dropped


2025-03-14 19:05:25,700 - eeg_notebook - INFO - Creadas 29 épocas con 64 canales
2025-03-14 19:05:25,704 - eeg_notebook - INFO - Datos extraídos: X shape (29, 51264), y shape (29,)


  Sujeto: 96, Tarea: motor_imagery, Paradigma: left_right_hand
  Forma de datos: (29, 51264)
  Clases: ['rest', 'left_hand', 'right_hand']
  Distribución de clases: {'rest': 14, 'left_hand': 8, 'right_hand': 7}

Cargados 6 EEGs exitosamente.

Resumen de datos EEG:
   Subject        Task Type         Paradigm  Num Samples  Num Features  \
0       61    motor_imagery       hands_feet           29         51264   
1       74  motor_execution       hands_feet           28         51264   
2      106    motor_imagery       hands_feet           29         51264   
3       34    motor_imagery       hands_feet           28         51264   
4       51  motor_execution  left_right_hand           28         51264   
5       96    motor_imagery  left_right_hand           29         51264   

                       Classes  
0  rest, both_hands, both_feet  
1  rest, both_hands, both_feet  
2  rest, both_hands, both_feet  
3  rest, both_hands, both_feet  
4  rest, left_hand, right_hand  
5  rest, le

2025-03-14 19:06:19,881 - eeg_notebook - ERROR - Error en el experimento: cannot reshape array of size 816 into shape (136,64,0)


ValueError: cannot reshape array of size 816 into shape (136,64,0)